PHASE-2

In [8]:
import pandas as pd
import numpy as np

# Load Phase 0 clean file
df = pd.read_csv("../data/processed/products_clean.csv")

In [9]:
# Step 2.1 — Parse prices & Clean Metrics

# Defensively clean currency symbols/commas if they still exist, then convert to float
for col in ['discounted_price', 'actual_price']:
    df[col] = df[col].astype(str).str.replace('₹', '', regex=False).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Parse rating_count to numeric (Fixing commas like "24,152")
df['rating_count'] = df['rating_count'].astype(str).str.replace(',', '', regex=False)
df['rating_count'] = pd.to_numeric(df['rating_count'], errors='coerce').fillna(0).astype(int)

# Sanity check: does actual_price * (1 - discount_percentage) roughly equal discounted_price?
df['implied_discounted'] = df['actual_price'] * (1 - df['discount_percentage'])
df['price_check_diff'] = (df['implied_discounted'] - df['discounted_price']).abs()


print(df['price_check_diff'].describe())


count    1464.000000
mean       11.345502
std        29.834781
min         0.000000
25%         0.650000
50%         2.420000
75%         8.682500
max       354.000000
Name: price_check_diff, dtype: float64


In [10]:
print((df['price_check_diff'] > 50).sum(), "products with diff > ₹50")
print(df[df['price_check_diff'] > 50][['product_id','actual_price','discounted_price','discount_percentage']].head())

72 products with diff > ₹50
    product_id  actual_price  discounted_price  discount_percentage
19  B08DPLCM6T       21990.0           13490.0                 0.39
24  B0B1YVCJ2Y       19990.0           11499.0                 0.42
38  B0B3MMYHYW       45999.0           32999.0                 0.28
57  B09Q5SWVBJ       21999.0           15999.0                 0.27
61  B0B15CPR37       47900.0           32990.0                 0.31


In [11]:
# Step 2.2 — Split the category hierarchy

df['category_levels'] = df['category'].str.split('|').fillna('').apply(len)

cat_split = df['category'].str.split('|', expand=True)
df['category_main'] = cat_split[0].fillna('Unknown')
df['category_sub'] = cat_split[1].fillna('Unknown')

In [12]:
# Step 2.3 — Add price-tier bucketing

df['price_tier'] = pd.qcut(df['actual_price'], q=4, labels=['Budget','Mid','Premium','Luxury'])

In [13]:
# Re-calculate the reviewer count 
df['n_reviewers'] = df['user_id'].astype(str).str.split(',').map(len)

In [14]:
# Aggregate everything per product_id — run this AFTER prices/category/price_tier/n_reviewers exist on df
# creating star schema

agg_numeric = df.groupby('product_id').agg(
    product_name=('product_name', 'first'),
    category_main=('category_main', 'first'),
    category_sub=('category_sub', 'first'),
    actual_price=('actual_price', 'first'),
    discounted_price=('discounted_price', 'first'),
    discount_percentage=('discount_percentage', 'first'),
    rating=('rating', 'mean'),
    rating_count=('rating_count', 'max'),
    price_tier=('price_tier', 'first'),
    n_reviewers=('n_reviewers', 'sum'),   # sum — each duplicate snapshot is a distinct reviewer batch
).reset_index()

agg_text = df.groupby('product_id').agg(
    review_title=('review_title', lambda x: ','.join(x.dropna().astype(str))),
    review_content=('review_content', lambda x: ' '.join(x.dropna().astype(str))),
).reset_index()

df_dedup = agg_numeric.merge(agg_text, on='product_id')

dim_category = df_dedup[['category_main','category_sub']].drop_duplicates().reset_index(drop=True)
dim_category['category_id'] = dim_category.index + 1
df_dedup = df_dedup.merge(dim_category, on=['category_main','category_sub'], how='left')

dim_product = df_dedup[['product_id','product_name','category_id','price_tier']]
fact_product_metrics = df_dedup[['product_id','actual_price','discounted_price','discount_percentage','rating','rating_count','n_reviewers']]

# Review text stays Python-only — kept OUT of the Power BI model to avoid bloating the .pbix with long text blobs
review_text = df_dedup[['product_id','review_title','review_content']]

dim_category.to_csv("../data/processed/dim_category.csv", index=False)
dim_product.to_csv("../data/processed/dim_product.csv", index=False)
fact_product_metrics.to_csv("../data/processed/fact_product_metrics.csv", index=False)
review_text.to_csv("../data/processed/review_text.csv", index=False)

print("Dim Category:", dim_category.shape)
print("Dim Product:", dim_product.shape)
print("Fact Table:", fact_product_metrics.shape)
print("Review text:", review_text.shape)

Dim Category: (29, 3)
Dim Product: (1350, 4)
Fact Table: (1350, 7)
Review text: (1350, 3)
